# LLM/RAG 기반 가구 추천 챗봇 프로젝트

- 프로젝트명: 가구 이미지 분류 결과를 활용한 RAG 기반 가구 추천 챗봇
- 작성자: 모하영
- 목적: 딥러닝 이미지 분류 결과와 RAG 검색 결과를 결합하여 사용자 질문에 맞는 가구 추천 답변을 생성한다.

이 노트북은 LLM 프로젝트의 제출용 흐름 정리 파일입니다. 모델을 새로 학습하는 파일이 아니라, RAG가 어떤 방식으로 문서를 읽고 검색하며 답변 근거를 만드는지 확인하는 용도입니다.

## 1. 프로젝트 개요

기존 딥러닝 프로젝트에서는 가구 이미지를 침대, 의자, 소파, 테이블, 수납장 중 하나로 분류하는 CNN 모델을 구현하였다. 이번 LLM 프로젝트에서는 해당 이미지 분류 결과를 활용하여 사용자가 입력한 질문과 관련된 가구 추천 문서를 검색하고, 검색된 근거를 바탕으로 추천 답변을 생성하는 RAG 구조를 구성하였다.

- 딥러닝: 업로드된 이미지의 가구 카테고리 예측
- RAG: 예측 카테고리와 질문에 맞는 추천 문서 검색
- LLM 역할: 검색된 근거를 기반으로 사용자에게 자연스러운 추천 답변 생성

현재 노트북에서는 외부 LLM API 대신, RAG 검색 결과를 템플릿 답변으로 정리하여 전체 처리 흐름을 확인한다.

## 2. 라이브러리 불러오기

RAG 검색에는 문서를 숫자 벡터로 변환하는 `TfidfVectorizer`와 문서 유사도를 계산하는 `cosine_similarity`를 사용한다.

In [ ]:
import re
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 3. 프로젝트 경로 설정

노트북이 LLM 프로젝트 폴더 안에 있으면 현재 폴더를 사용하고, 그렇지 않으면 `C:\\AIProject\\llm_mohayoung_project` 경로를 사용한다.

In [ ]:
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "furniture_knowledge.md").exists():
    PROJECT_DIR = Path(r"C:\AIProject\llm_mohayoung_project")

KNOWLEDGE_PATH = PROJECT_DIR / "data" / "furniture_knowledge.md"
DL_MODEL_PATH = PROJECT_DIR.parent / "dl_furniture_project" / "models" / "furniture_cnn.keras"

print("LLM 프로젝트 경로:", PROJECT_DIR)
print("RAG 지식 문서:", KNOWLEDGE_PATH)
print("지식 문서 존재 여부:", KNOWLEDGE_PATH.exists())
print("딥러닝 모델 존재 여부:", DL_MODEL_PATH.exists())

## 4. RAG 지식 문서 확인

`furniture_knowledge.md` 파일은 가구 카테고리별 추천 기준과 배치 가이드를 담고 있다. RAG는 이 문서를 검색 대상으로 사용한다.

In [ ]:
knowledge_text = KNOWLEDGE_PATH.read_text(encoding="utf-8")

print("문서 길이:", len(knowledge_text))
print(knowledge_text[:500])

## 5. RAG 문서 로드 및 검색 클래스 정의

문서를 `## 제목 [카테고리]` 단위로 나누어 검색 가능한 문서 목록으로 변환한다. 이후 TF-IDF 방식으로 문서를 벡터화하고, 사용자 질문과 문서 사이의 코사인 유사도를 계산한다.

In [ ]:
@dataclass
class RAGDocument:
    title: str
    category: str
    content: str


class FurnitureRAG:
    def __init__(self, knowledge_path):
        self.knowledge_path = Path(knowledge_path)
        self.documents = self._load_documents()
        self.vectorizer = TfidfVectorizer()
        self.doc_texts = [
            f"{doc.title} {doc.category} {doc.content}" for doc in self.documents
        ]
        self.matrix = self.vectorizer.fit_transform(self.doc_texts)

    def _load_documents(self):
        text = self.knowledge_path.read_text(encoding="utf-8")
        sections = re.split(r"\n## ", text)
        documents = []

        for raw_section in sections:
            section = raw_section.strip()
            if not section:
                continue

            lines = section.splitlines()
            title_line = lines[0].replace("## ", "").strip()
            if title_line.startswith("# "):
                continue

            content = "\n".join(lines[1:]).strip()
            match = re.search(r"\[(.*?)\]", title_line)
            category = match.group(1) if match else "general"

            documents.append(
                RAGDocument(
                    title=title_line,
                    category=category,
                    content=content,
                )
            )

        return documents

    def retrieve(self, query, category=None, top_k=3):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix)[0]

        ranked = []
        for idx, score in enumerate(scores):
            doc = self.documents[idx]
            category_bonus = 0.2 if category and doc.category == category else 0
            ranked.append((score + category_bonus, score, doc))

        ranked.sort(key=lambda item: item[0], reverse=True)
        return ranked[:top_k]

## 6. RAG 문서 목록 확인

RAG가 검색할 수 있는 문서가 몇 개인지 확인하고, 문서 제목과 카테고리를 표로 확인한다.

In [ ]:
rag = FurnitureRAG(KNOWLEDGE_PATH)

documents_df = pd.DataFrame(
    [
        {
            "title": doc.title,
            "category": doc.category,
            "content_preview": doc.content[:80],
        }
        for doc in rag.documents
    ]
)

print("검색 문서 수:", len(rag.documents))
documents_df

## 7. 사용자 질문 기반 문서 검색

사용자 질문과 이미지 분류 결과를 함께 사용하여 관련 문서를 검색한다. 예를 들어 이미지 분류 결과가 `couch`이고 사용자가 거실 배치를 질문하면, 소파 관련 문서가 우선 검색된다.

In [ ]:
sample_category_en = "couch"
sample_category_ko = "소파"
sample_question = "작은 거실에 어울리는 배치와 색상을 추천해줘"

retrieved = rag.retrieve(
    query=f"{sample_category_ko} {sample_category_en} {sample_question}",
    category=sample_category_en,
    top_k=3,
)

retrieved_df = pd.DataFrame(
    [
        {
            "rank": idx + 1,
            "final_score": round(final_score, 4),
            "similarity": round(similarity, 4),
            "title": doc.title,
            "category": doc.category,
            "content": doc.content[:160],
        }
        for idx, (final_score, similarity, doc) in enumerate(retrieved)
    ]
)

retrieved_df

## 8. 검색 결과 기반 답변 생성

실제 LLM API를 연결하면 검색된 문서를 프롬프트에 넣어 자연스러운 답변을 생성할 수 있다. 현재 노트북에서는 외부 API 없이 검색된 근거를 기반으로 템플릿 답변을 생성한다.

In [ ]:
def generate_rag_answer(category_ko, category_en, question, retrieved_items):
    evidence = "\n".join(
        f"- {doc.title}: {doc.content}" for _, _, doc in retrieved_items
    )

    return f"""
현재 업로드 이미지의 예측 카테고리는 {category_ko}({category_en})입니다.

사용자 질문: {question}

검색된 추천 근거:
{evidence}

추천 답변:
현재 조건에서는 {category_ko} 계열 상품을 중심으로 공간 크기, 사용 목적, 색상 조합을 함께 고려하는 것이 좋습니다.
특히 검색된 문서에서 확인한 것처럼 배치 위치와 주변 가구와의 조화가 중요하므로, 실제 서비스에서는 이 근거를 바탕으로 사용자에게 더 자연스러운 추천 문장을 제공할 수 있습니다.
""".strip()


answer = generate_rag_answer(
    sample_category_ko,
    sample_category_en,
    sample_question,
    retrieved,
)

print(answer)

## 9. 딥러닝 이미지 분류 모델과의 연결 구조

LLM 프로젝트는 딥러닝 모델을 새로 학습하지 않고, 기존 딥러닝 프로젝트에서 저장한 CNN 모델을 불러와 이미지 카테고리를 예측한다. 그 예측 결과가 RAG 검색의 카테고리 힌트로 사용된다.

In [ ]:
print("딥러닝 모델 경로:", DL_MODEL_PATH)
print("딥러닝 모델 파일 존재 여부:", DL_MODEL_PATH.exists())

print("이미지 예측 코드 경로:", PROJECT_DIR / "image_predictor.py")
print("이미지 예측 코드 존재 여부:", (PROJECT_DIR / "image_predictor.py").exists())

## 10. 전체 처리 흐름 정리

```text
사용자 이미지 업로드
→ CNN 모델이 가구 카테고리 예측
→ 예측 카테고리와 사용자 질문 결합
→ RAG가 관련 추천 문서 검색
→ 검색된 근거를 기반으로 추천 답변 생성
→ Streamlit 화면에서 결과 시연
```

이 구조는 기존 웹 프로젝트의 상품 검색, 추천, 챗봇 기능과 연결할 수 있다. 예를 들어 사용자가 가구 이미지를 업로드하면 이미지 분류 모델이 카테고리를 추정하고, RAG는 해당 카테고리에 맞는 추천 설명과 배치 가이드를 제공할 수 있다.

## 11. 한계점 및 개선 방향

- 현재 이미지 분류 모델은 CIFAR-100 기반 32x32 이미지로 학습되어 실제 고해상도 상품 이미지에서는 예측 정확도가 낮아질 수 있다.
- RAG 문서는 직접 작성한 소규모 문서이므로 추천 근거의 다양성이 부족할 수 있다.
- 현재 노트북은 외부 LLM API 없이 템플릿 기반 답변을 사용한다.
- 향후 Kaggle 또는 Colab에서 고해상도 가구 이미지 데이터셋으로 전이학습 모델을 학습하면 이미지 분류 품질을 개선할 수 있다.
- LangChain, Vector DB, OpenAI API 등을 연결하면 실제 LLM 기반 자연어 답변 생성 구조로 확장할 수 있다.